In [2]:
import pandas

In [3]:
cell_df = pandas.read_csv("smo_data/cell_type_classification.csv", index_col=0)

In [4]:
mito_inh_df = pandas.read_csv("smo_data/inh_mito_fullstats.csv", index_col=0)


In [5]:
mito_inh_df.keys()

Index(['mito_vx', 'ctr_pos_x_vx', 'ctr_pos_y_vx', 'ctr_pos_z_vx',
       'bbox_beg_x_vx', 'bbox_beg_y_vx', 'bbox_beg_z_vx', 'bbox_end_x_vx',
       'bbox_end_y_vx', 'bbox_end_z_vx', 'cellid', 'ctr_pos_x_nm',
       'ctr_pos_y_nm', 'ctr_pos_z_nm', 'surface_area', 'complexityindex',
       'compartment', 'pathlength'],
      dtype='object')

In [6]:
mito_inh_df.groupby(by="compartment")[["pathlength"]].describe()

pathlength                                             \
                 count         mean          std  min         25%   
compartment                                                         
Axonal          7212.0  1130.875501  1091.611841  0.0  583.589085   
Basal           8663.0  2199.828546  2096.032276  0.0  900.866338   
Somatic        11437.0  1111.056257  2509.114684  0.0    0.000000   

                                                     
                     50%          75%           max  
compartment                                          
Axonal        964.570294  1383.589312  27961.982908  
Basal        1591.255663  2796.035196  34469.268709  
Somatic         0.000000     0.000000  25678.579957

In [7]:
selected_columns = ["cellid", "compartment"]

mito_inh_df = mito_inh_df[selected_columns]

In [8]:
mito_inh_morpho_df = pandas.read_csv("smo_data/inh_mito_morphometrics.csv", index_col=0)

In [9]:
mito_inh_df = mito_inh_df.join(mito_inh_morpho_df, how="inner")

In [10]:

df = mito_inh_df.reset_index()\
    .merge(right=cell_df[["cell_type", "cell_subtype", "cell_segid"]], left_on="cellid", right_on="cell_segid")

df=df.drop(columns="cell_segid")

In [11]:
df

,mito_id,cellid,compartment,mito size,mito SA,PC1 Length,PC2 Length,PC3 Length,PC1 inertia moment,PC2 inertia moment,...,PC2 symmetry,PC3 symmetry,sphericity,convex hull volume,convex hull SA,convex hull compactness,mito CA,mito diam,cell_type,cell_subtype
0,2639184,648518346349528994,Basal,2.132590e+07,7.189440e+05,1466.911721,316.149262,250.974113,2.203152e+06,4.627762e+07,...,0.888722,0.973477,0.517282,4.568215e+07,9.192166e+05,0.466832,1.621513e+05,1466.911721,inhibitory,basket
1,2634928,648518346349528994,Basal,1.010730e+09,1.893197e+07,10622.711429,2042.528186,922.304883,1.505722e+09,4.263860e+10,...,0.899397,0.852742,0.257264,7.725862e+09,3.451310e+07,0.130824,2.553243e+06,10622.711429,inhibitory,basket
2,2626714,648518346349528994,Somatic,4.047698e+07,9.853451e+05,1458.297382,437.332323,288.639328,5.671749e+06,7.325149e+07,...,0.875640,0.995875,0.578586,7.607299e+07,1.238109e+06,0.532081,2.473207e+05,1458.297382,inhibitory,basket
3,1610907,648518346349528994,Axonal,9.092232e+07,1.418191e+06,1193.034143,489.626357,376.449781,1.964063e+07,5.343744e+07,...,0.987538,0.999391,0.689493,1.088547e+08,1.367225e+06,0.835263,3.902470e+05,1193.034143,inhibitory,basket
4,2120388,648518346349528994,Basal,2.637172e+08,5.596541e+06,7297.389393,909.058805,621.881404,1.232018e+08,9.168005e+09,...,0.791932,0.998936,0.355350,1.756105e+09,1.282820e+07,0.150172,8.692413e+05,7297.389393,inhibitory,basket
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27307,3647060,648518346349537389,Somatic,2.124093e+08,3.554721e+06,2032.045682,1241.833018,535.887504,1.075943e+08,3.203622e+08,...,0.831938,0.947431,0.484315,6.860911e+08,5.183335e+06,0.309593,8.676042e+05,2032.045682,inhibitory,bipolar
27308,3619006,648518346349537389,Basal,5.004799e+08,9.030464e+06,8772.446007,1329.198406,1117.863022,3.450557e+08,1.604025e+10,...,0.979574,0.928752,0.337571,4.863070e+09,2.371929e+07,0.102914,6.153581e+05,8772.446007,inhibitory,bipolar
27309,3779882,648518346349537389,Basal,2.609691e+08,3.381068e+06,2097.834520,1160.923425,623.266843,8.199197e+07,3.007443e+08,...,0.812229,0.940292,0.584103,5.068589e+08,4.061400e+06,0.514875,7.357306e+05,2097.834520,inhibitory,bipolar
27310,3610114,648518346349537389,Basal,2.715740e+08,4.728983e+06,3883.910918,1005.920461,548.082959,5.742912e+07,1.705759e+09,...,0.987402,0.997058,0.428853,8.230312e+08,6.705953e+06,0.329968,1.059694e+06,3883.910918,inhibitory,bipolar


In [12]:
cell_ids = df["cellid"].unique()

In [13]:
len(cell_ids)

33

In [15]:
!mkdir -p smo_data/microns_cells

In [14]:
def merge_mitodfs(mito_file_with_compartment_label, mito_file_with_morphometrics, celltypefile):
    cell_df = pandas.read_csv(celltypefile, index_col=0)
    mito_df = pandas.read_csv(mito_file_with_compartment_label, index_col=0)


    
    VOXELRES = [3.58, 3.58, 40.0]  
    for c, s in zip("xyz", VOXELRES):
        mito_df[c] = mito_df["ctr_pos_{:s}_vx".format(c)]*s

    selected_columns = [
        "cellid",
        "compartment",
        "x", #check whether they fit with the skeleton
        "y",
        "z",
        "pathlength",
    ]
    mito_df = mito_df[selected_columns]
    mito_df = mito_df.rename(columns={"pathlength": "length along skeleton"})
    mito_morpho_df = pandas.read_csv(mito_file_with_morphometrics, index_col=0)

    mito__df = mito_df.join(mito_morpho_df, how="inner")


    df = mito__df.reset_index()\
        .merge(right=cell_df[["cell_type", "cell_subtype", "cell_segid"]], left_on="cellid", right_on="cell_segid")

    df=df.drop(columns="cell_segid")
    return df

def write_cellfiles(mito_file_with_compartment_label, mito_file_with_morphometrics, celltypefile):
    df = merge_mitodfs(mito_file_with_compartment_label, mito_file_with_morphometrics, celltypefile)

    cell_ids = df["cellid"].unique()
    
    for cell_id in cell_ids:
        dfh=df.query("cellid==@cell_id").reset_index(drop=True)
        celltype = dfh.iloc[0]["cell_type"]
        filename = "smo_data/microns_cells/{:s}_{:d}.csv".format(celltype, int(cell_id))
        print(filename)
        dfh.to_csv(filename)

In [17]:
write_cellfiles("smo_data/inh_mito_fullstats.csv", "smo_data/inh_mito_morphometrics.csv", "smo_data/cell_type_classification.csv")

smo_data/microns_cells/inhibitory_648518346349528994.csv
smo_data/microns_cells/inhibitory_648518346349525188.csv
smo_data/microns_cells/inhibitory_648518346349538791.csv
smo_data/microns_cells/inhibitory_648518346349538285.csv
smo_data/microns_cells/inhibitory_648518346349538789.csv
smo_data/microns_cells/inhibitory_648518346349488919.csv
smo_data/microns_cells/inhibitory_648518346349517783.csv
smo_data/microns_cells/inhibitory_648518346349518096.csv
smo_data/microns_cells/inhibitory_648518346349489861.csv
smo_data/microns_cells/inhibitory_648518346349515985.csv
smo_data/microns_cells/inhibitory_648518346349539846.csv
smo_data/microns_cells/inhibitory_648518346349489985.csv
smo_data/microns_cells/inhibitory_648518346349539215.csv
smo_data/microns_cells/inhibitory_648518346349536849.csv
smo_data/microns_cells/inhibitory_648518346349522750.csv
smo_data/microns_cells/inhibitory_648518346349525190.csv
smo_data/microns_cells/inhibitory_648518346349538638.csv
smo_data/microns_cells/inhibito

In [18]:
write_cellfiles("data/pni_mito_analysisids_fullstats.csv", "smo_data/pyr_mito_morphometrics.csv", "smo_data/cell_type_classification.csv")

smo_data/microns_cells/pyramidal_648518346349491311.csv
smo_data/microns_cells/pyramidal_648518346349492130.csv
smo_data/microns_cells/pyramidal_648518346349492197.csv
smo_data/microns_cells/pyramidal_648518346349492682.csv
smo_data/microns_cells/pyramidal_648518346349493472.csv
smo_data/microns_cells/pyramidal_648518346349493487.csv
smo_data/microns_cells/pyramidal_648518346349493874.csv
smo_data/microns_cells/pyramidal_648518346349494004.csv
smo_data/microns_cells/pyramidal_648518346349494577.csv
smo_data/microns_cells/pyramidal_648518346349503588.csv
smo_data/microns_cells/pyramidal_648518346349505739.csv
smo_data/microns_cells/pyramidal_648518346349507351.csv
smo_data/microns_cells/pyramidal_648518346349517132.csv
smo_data/microns_cells/pyramidal_648518346349519354.csv
smo_data/microns_cells/pyramidal_648518346349520120.csv
smo_data/microns_cells/pyramidal_648518346349521083.csv
smo_data/microns_cells/pyramidal_648518346349522230.csv
smo_data/microns_cells/pyramidal_648518346349522

smo_data/microns_cells/pyramidal_648518346349537649.csv
smo_data/microns_cells/pyramidal_648518346349537657.csv
smo_data/microns_cells/pyramidal_648518346349537687.csv
smo_data/microns_cells/pyramidal_648518346349537692.csv
smo_data/microns_cells/pyramidal_648518346349537716.csv
smo_data/microns_cells/pyramidal_648518346349537717.csv
smo_data/microns_cells/pyramidal_648518346349537718.csv
smo_data/microns_cells/pyramidal_648518346349537741.csv
smo_data/microns_cells/pyramidal_648518346349537790.csv
smo_data/microns_cells/pyramidal_648518346349537798.csv
smo_data/microns_cells/pyramidal_648518346349537808.csv
smo_data/microns_cells/pyramidal_648518346349537814.csv
smo_data/microns_cells/pyramidal_648518346349537818.csv
smo_data/microns_cells/pyramidal_648518346349537827.csv
smo_data/microns_cells/pyramidal_648518346349537828.csv
smo_data/microns_cells/pyramidal_648518346349537835.csv
smo_data/microns_cells/pyramidal_648518346349537844.csv
smo_data/microns_cells/pyramidal_648518346349537

smo_data/microns_cells/pyramidal_648518346349539821.csv
smo_data/microns_cells/pyramidal_648518346349539825.csv
smo_data/microns_cells/pyramidal_648518346349539828.csv
smo_data/microns_cells/pyramidal_648518346349539832.csv
smo_data/microns_cells/pyramidal_648518346349539834.csv
smo_data/microns_cells/pyramidal_648518346349539836.csv
smo_data/microns_cells/pyramidal_648518346349539840.csv
smo_data/microns_cells/pyramidal_648518346349539844.csv
smo_data/microns_cells/pyramidal_648518346349539845.csv
smo_data/microns_cells/pyramidal_648518346349539851.csv
smo_data/microns_cells/pyramidal_648518346349539852.csv
smo_data/microns_cells/pyramidal_648518346349539853.csv
smo_data/microns_cells/pyramidal_648518346349539856.csv
smo_data/microns_cells/pyramidal_648518346349539862.csv
smo_data/microns_cells/pyramidal_648518346349539863.csv
smo_data/microns_cells/pyramidal_648518346349539864.csv
smo_data/microns_cells/pyramidal_648518346349539865.csv
smo_data/microns_cells/pyramidal_648518346349539

In [42]:
write_cellfiles("smo_data/astro_mito_fullstats.csv",
               "smo_data/astro_mito_morphometrics.csv",
               "smo_data/cell_type_classification.csv")

smo_data/microns_cells/glia_648518346349536487.csv
smo_data/microns_cells/glia_648518346349527316.csv
smo_data/microns_cells/glia_648518346349530569.csv
smo_data/microns_cells/glia_648518346349386860.csv
smo_data/microns_cells/glia_648518346349517141.csv
smo_data/microns_cells/glia_648518346342795947.csv
smo_data/microns_cells/glia_648518346349528250.csv
smo_data/microns_cells/glia_648518346349525544.csv
smo_data/microns_cells/glia_648518346349498574.csv
smo_data/microns_cells/glia_648518346349525862.csv
smo_data/microns_cells/glia_648518346349527319.csv
smo_data/microns_cells/glia_648518346349528249.csv
smo_data/microns_cells/glia_648518346349521344.csv
smo_data/microns_cells/glia_648518346349524139.csv


In [15]:
pyr_df = merge_mitodfs("data/pni_mito_analysisids_fullstats.csv", 
                       "smo_data/pyr_mito_morphometrics.csv", 
                       "smo_data/cell_type_classification.csv")

In [16]:
inh_df = merge_mitodfs("smo_data/inh_mito_fullstats.csv", 
                       "smo_data/inh_mito_morphometrics.csv", 
                       "smo_data/cell_type_classification.csv")

In [17]:
astro_df = merge_mitodfs("smo_data/astro_mito_fullstats.csv", "smo_data/astro_mito_morphometrics.csv", "smo_data/cell_type_classification.csv")

In [18]:
df = pandas.concat((pyr_df, inh_df, astro_df))

In [38]:
df.keys()

Index(['mito_id', 'cellid', 'compartment', 'x', 'y', 'z',
       'length along skeleton', 'PC1 CA', 'PC1 Circum', 'PC1 Length',
       'PC1 inertia moment', 'PC1 symmetry', 'PC2 CA', 'PC2 Circum',
       'PC2 Length', 'PC2 inertia moment', 'PC2 symmetry', 'PC3 CA',
       'PC3 Circum', 'PC3 Length', 'PC3 inertia moment', 'PC3 symmetry',
       'convex hull SA', 'convex hull compactness', 'convex hull volume',
       'mito CA', 'mito SA', 'mito diam', 'mito size', 'sphericity',
       'cell_type', 'cell_subtype'],
      dtype='object')

In [40]:
df.groupby(by=["cell_type"])[["cellid"]].apply(lambda sr: len(sr["cellid"].unique()))

cell_type
glia           14
inhibitory     33
pyramidal     351
dtype: int64

In [41]:
df.groupby(by=["cell_type", "compartment"])["mito_id"].count()

cell_type   compartment      
glia        Branch                9128
            Far                    242
            Soma                  1207
inhibitory  Axonal                7212
            Basal                 8663
            Somatic              11437
pyramidal   Apical               18608
            Axonal               11484
            Basal                53318
            Somatic              90193
            Unknown               1811
            Unknown dendritic      658
Name: mito_id, dtype: int64

In [51]:
hdf = pyr_df
idx = pyr_df["PC1 Length"].idxmax()
cell_id = hdf.loc[idx]["cellid"]
mitoids = hdf.loc[hdf["cellid"]==cell_id]["mito_id"].values
print(cell_id)
print(" ".join([str(m) for m in mitoids]))

648518346349538416
1764183 3111024 3115079 3315094 3002675 2895331 3007851 3007365 3226561 3007915 2852120 2997237 3111351 3002698 3001331 3446065 2794891 3115817 3115468 3321680 4071031 3007903 1950403 3007822 3007698 3752968 3007883 3007897 1585874 3115242 3012777 3002707 3116154 2306803 3007135 3006431 3012760 3007891 3002653 3007347 3007533 3116095 3110317 1230989 3115206 3116397 2669549 3116425 3007286 3007617 2665516 3067198 3006881 3230145 3007271 3554746 3001811 3115198 1350547 3002476 3002548 3001344 3753674 1011018 3116464 2978599 3007451 3058542 3054384 3120263 2895093 3115460 3001359 3120384 3091032 3115120 2983489 3328189 2992460 3537194 3007693 3209132 3002565 3115593 3002643 3236267 2992300 3007806 3006517 4226879 3006972 3226734 2904197 3002549 2663772 3002350 3115606 2128185 3331356 3115769 3120369 3115718 3116077 3115309 3002686 3115784 3261491 3007017 3001938 3001406 2978351 3115901 3277220 2187581 2988300 3517989 2620077 3968571 3115853 3071405 3115612 3116092 30074

[Excitatory cell](https://neuromancer-seung-import.appspot.com/#!%7B%22layers%22:%5B%7B%22source%22:%22precomputed://gs://microns_public_datasets/pinky100_v0/son_of_alignment_v15_rechunked%22%2C%22type%22:%22image%22%2C%22blend%22:%22default%22%2C%22shaderControls%22:%7B%7D%2C%22name%22:%22EM%22%7D%2C%7B%22source%22:%22precomputed://gs://microns_public_datasets/pinky100_v185/seg%22%2C%22type%22:%22segmentation%22%2C%22selectedAlpha%22:0.51%2C%22objectAlpha%22:0.09%2C%22segmentColors%22:%7B%22648518346349538416%22:%22#1c71d8%22%7D%2C%22segments%22:%5B%22648518346349538416%22%5D%2C%22skeletonRendering%22:%7B%22mode2d%22:%22lines_and_points%22%2C%22mode3d%22:%22lines%22%7D%2C%22name%22:%22cell_segmentation_v185%22%7D%2C%7B%22source%22:%22precomputed://https://td.princeton.edu/sseung-archive/pinky100-mito/seg_191220%22%2C%22type%22:%22segmentation%22%2C%22segments%22:%5B%221009302%22%2C%221011018%22%2C%221120502%22%2C%221121159%22%2C%221230989%22%2C%221341592%22%2C%221343017%22%2C%221349554%22%2C%221350547%22%2C%221568592%22%2C%221585874%22%2C%221764183%22%2C%221889599%22%2C%221950039%22%2C%221950403%22%2C%222058297%22%2C%222058760%22%2C%222116878%22%2C%222127556%22%2C%222127779%22%2C%222128185%22%2C%222132671%22%2C%222162375%22%2C%222182506%22%2C%222183401%22%2C%222187510%22%2C%222187581%22%2C%222187804%22%2C%222243386%22%2C%222250456%22%2C%222279940%22%2C%222306803%22%2C%222323572%22%2C%222323727%22%2C%222324500%22%2C%222372469%22%2C%222398766%22%2C%222429867%22%2C%222441438%22%2C%222442037%22%2C%222442219%22%2C%222447259%22%2C%222515754%22%2C%222546492%22%2C%222557712%22%2C%222559199%22%2C%222611754%22%2C%222614714%22%2C%222620077%22%2C%222629297%22%2C%222663772%22%2C%222665516%22%2C%222669549%22%2C%222674089%22%2C%222674880%22%2C%222679290%22%2C%222740951%22%2C%222782805%22%2C%222782981%22%2C%222788605%22%2C%222794891%22%2C%222840457%22%2C%222852120%22%2C%222862913%22%2C%222893625%22%2C%222893677%22%2C%222893797%22%2C%222893829%22%2C%222893836%22%2C%222893973%22%2C%222894868%22%2C%222895079%22%2C%222895093%22%2C%222895331%22%2C%222898733%22%2C%222900560%22%2C%222904060%22%2C%222904197%22%2C%222904251%22%2C%222950827%22%2C%222974361%22%2C%222974522%22%2C%222975732%22%2C%222977676%22%2C%222978013%22%2C%222978233%22%2C%222978312%22%2C%222978351%22%2C%222978599%22%2C%222983359%22%2C%222983413%22%2C%222983489%22%2C%222988236%22%2C%222988300%22%2C%222992300%22%2C%222992302%22%2C%222992397%22%2C%222992401%22%2C%222992433%22%2C%222992460%22%2C%222996994%22%2C%222997061%22%2C%222997201%22%2C%222997213%22%2C%222997237%22%2C%222997252%22%2C%222997373%22%2C%222997434%22%2C%223001331%22%2C%223001344%22%2C%223001359%22%2C%223001406%22%2C%223001483%22%2C%223001489%22%2C%223001598%22%2C%223001623%22%2C%223001627%22%2C%223001755%22%2C%223001756%22%2C%223001811%22%2C%223001938%22%2C%223001955%22%2C%223001958%22%2C%223001959%22%2C%223001995%22%2C%223002039%22%2C%223002050%22%2C%223002121%22%2C%223002138%22%2C%223002154%22%2C%223002159%22%2C%223002236%22%2C%223002240%22%2C%223002306%22%2C%223002319%22%2C%223002323%22%2C%223002349%22%2C%223002350%22%2C%223002385%22%2C%223002390%22%2C%223002426%22%2C%223002452%22%2C%223002454%22%2C%223002473%22%2C%223002476%22%2C%223002516%22%2C%223002548%22%2C%223002549%22%2C%223002565%22%2C%223002643%22%2C%223002653%22%2C%223002668%22%2C%223002675%22%2C%223002686%22%2C%223002698%22%2C%223002707%22%2C%223005991%22%2C%223006043%22%2C%223006094%22%2C%223006151%22%2C%223006172%22%2C%223006175%22%2C%223006187%22%2C%223006193%22%2C%223006198%22%2C%223006209%22%2C%223006241%22%2C%223006295%22%2C%223006301%22%2C%223006302%22%2C%223006309%22%2C%223006324%22%2C%223006339%22%2C%223006371%22%2C%223006377%22%2C%223006420%22%2C%223006431%22%2C%223006470%22%2C%223006505%22%2C%223006517%22%2C%223006594%22%2C%223006607%22%2C%223006609%22%2C%223006641%22%2C%223006669%22%2C%223006707%22%2C%223006708%22%2C%223006722%22%2C%223006729%22%2C%223006818%22%2C%223006833%22%2C%223006881%22%2C%223006887%22%2C%223006919%22%2C%223006967%22%2C%223006972%22%2C%223007017%22%2C%223007028%22%2C%223007105%22%2C%223007122%22%2C%223007130%22%2C%223007135%22%2C%223007147%22%2C%223007156%22%2C%223007190%22%2C%223007193%22%2C%223007235%22%2C%223007239%22%2C%223007271%22%2C%223007277%22%2C%223007286%22%2C%223007309%22%2C%223007310%22%2C%223007320%22%2C%223007347%22%2C%223007359%22%2C%223007365%22%2C%223007397%22%2C%223007433%22%2C%223007451%22%2C%223007460%22%2C%223007464%22%2C%223007484%22%2C%223007498%22%2C%223007533%22%2C%223007544%22%2C%223007572%22%2C%223007610%22%2C%223007617%22%2C%223007644%22%2C%223007693%22%2C%223007698%22%2C%223007731%22%2C%223007737%22%2C%223007778%22%2C%223007797%22%2C%223007806%22%2C%223007813%22%2C%223007822%22%2C%223007841%22%2C%223007851%22%2C%223007883%22%2C%223007891%22%2C%223007897%22%2C%223007903%22%2C%223007915%22%2C%223012612%22%2C%223012668%22%2C%223012702%22%2C%223012760%22%2C%223012769%22%2C%223012777%22%2C%223012826%22%2C%223013005%22%2C%223017903%22%2C%223039805%22%2C%223044690%22%2C%223052574%22%2C%223052843%22%2C%223054384%22%2C%223056534%22%2C%223058010%22%2C%223058542%22%2C%223061796%22%2C%223067198%22%2C%223071405%22%2C%223071594%22%2C%223083449%22%2C%223090023%22%2C%223090216%22%2C%223090357%22%2C%223090550%22%2C%223091032%22%2C%223110270%22%2C%223110292%22%2C%223110317%22%2C%223110393%22%2C%223110455%22%2C%223110497%22%2C%223110516%22%2C%223110534%22%2C%223110563%22%2C%223110605%22%2C%223110776%22%2C%223110785%22%2C%223110799%22%2C%223110834%22%2C%223110887%22%2C%223111024%22%2C%223111326%22%2C%223111351%22%2C%223111444%22%2C%223115079%22%2C%223115094%22%2C%223115115%22%2C%223115119%22%2C%223115120%22%2C%223115126%22%2C%223115165%22%2C%223115174%22%2C%223115198%22%2C%223115206%22%2C%223115238%22%2C%223115241%22%2C%223115242%22%2C%223115250%22%2C%223115255%22%2C%223115284%22%2C%223115296%22%2C%223115309%22%2C%223115339%22%2C%223115352%22%2C%223115406%22%2C%223115411%22%2C%223115446%22%2C%223115460%22%2C%223115466%22%2C%223115468%22%2C%223115482%22%2C%223115488%22%2C%223115496%22%2C%223115524%22%2C%223115525%22%2C%223115540%22%2C%223115556%22%2C%223115559%22%2C%223115579%22%2C%223115589%22%2C%223115593%22%2C%223115598%22%2C%223115600%22%2C%223115603%22%2C%223115606%22%2C%223115612%22%2C%223115628%22%2C%223115649%22%2C%223115717%22%2C%223115718%22%2C%223115769%22%2C%223115784%22%2C%223115788%22%2C%223115796%22%2C%223115817%22%2C%223115821%22%2C%223115853%22%2C%223115876%22%2C%223115901%22%2C%223115904%22%2C%223115931%22%2C%223115946%22%2C%223115990%22%2C%223116020%22%2C%223116034%22%2C%223116071%22%2C%223116072%22%2C%223116077%22%2C%223116080%22%2C%223116090%22%2C%223116092%22%2C%223116095%22%2C%223116112%22%2C%223116114%22%2C%223116126%22%2C%223116154%22%2C%223116214%22%2C%223116217%22%2C%223116218%22%2C%223116230%22%2C%223116247%22%2C%223116260%22%2C%223116268%22%2C%223116282%22%2C%223116283%22%2C%223116287%22%2C%223116295%22%2C%223116397%22%2C%223116425%22%2C%223116464%22%2C%223116483%22%2C%223119759%22%2C%223119780%22%2C%223120073%22%2C%223120244%22%2C%223120255%22%2C%223120263%22%2C%223120369%22%2C%223120384%22%2C%223120537%22%2C%223121023%22%2C%223121295%22%2C%223173156%22%2C%223193213%22%2C%223208557%22%2C%223208607%22%2C%223208771%22%2C%223209047%22%2C%223209132%22%2C%223209904%22%2C%223225530%22%2C%223226561%22%2C%223226734%22%2C%223229686%22%2C%223229857%22%2C%223230060%22%2C%223230145%22%2C%223231172%22%2C%223231292%22%2C%223235865%22%2C%223236064%22%2C%223236151%22%2C%223236267%22%2C%223261491%22%2C%223277220%22%2C%223278237%22%2C%223299144%22%2C%223299612%22%2C%223300307%22%2C%223315094%22%2C%223315181%22%2C%223321401%22%2C%223321547%22%2C%223321680%22%2C%223328189%22%2C%223328327%22%2C%223331356%22%2C%223331655%22%2C%223332688%22%2C%223380918%22%2C%223382278%22%2C%223431605%22%2C%223444788%22%2C%223445158%22%2C%223446065%22%2C%223517460%22%2C%223517989%22%2C%223536840%22%2C%223537194%22%2C%223554746%22%2C%223555251%22%2C%223587919%22%2C%223645168%22%2C%223648442%22%2C%223649026%22%2C%223649180%22%2C%223649619%22%2C%223748211%22%2C%223752835%22%2C%223752968%22%2C%223753674%22%2C%223757549%22%2C%223864178%22%2C%223968571%22%2C%223968713%22%2C%223983463%22%2C%224071031%22%2C%224081318%22%2C%224100222%22%2C%224219818%22%2C%224220023%22%2C%224220254%22%2C%224226879%22%2C%224318839%22%2C%22897648%22%5D%2C%22skeletonRendering%22:%7B%22mode2d%22:%22lines_and_points%22%2C%22mode3d%22:%22lines%22%7D%2C%22name%22:%22mitochondria%22%7D%2C%7B%22source%22:%22precomputed://https://td.princeton.edu/sseung-archive/pinky100-nuclei/seg%22%2C%22type%22:%22segmentation%22%2C%22hiddenSegments%22:%5B%224308%22%5D%2C%22skeletonRendering%22:%7B%22mode2d%22:%22lines_and_points%22%2C%22mode3d%22:%22lines%22%7D%2C%22name%22:%22nuclei%22%7D%2C%7B%22source%22:%22precomputed://https://td.princeton.edu/sseung-archive/pinky100-clefts/mip1_d2_1175k%22%2C%22type%22:%22segmentation%22%2C%22skeletonRendering%22:%7B%22mode2d%22:%22lines_and_points%22%2C%22mode3d%22:%22lines%22%7D%2C%22name%22:%22synapses%22%2C%22visible%22:false%7D%5D%2C%22navigation%22:%7B%22pose%22:%7B%22position%22:%7B%22voxelSize%22:%5B4%2C4%2C40%5D%2C%22voxelCoordinates%22:%5B95842.3125%2C72268.375%2C1540.175537109375%5D%7D%7D%2C%22zoomFactor%22:49.21444684039923%7D%2C%22perspectiveOrientation%22:%5B0.0889565497636795%2C-0.12385793775320053%2C0.6608300805091858%2C-0.7348806262016296%5D%2C%22perspectiveZoom%22:3618.7659948513433%2C%22showSlices%22:false%2C%22selectedLayer%22:%7B%22layer%22:%22nuclei%22%2C%22visible%22:true%7D%2C%22layout%22:%7B%22type%22:%22xy-3d%22%2C%22orthographicProjection%22:true%7D%7D) in neuroglancer 

In [53]:
hdf = inh_df
idx = hdf["PC1 Length"].idxmax()
cell_id = hdf.loc[idx]["cellid"]
mitoids = hdf.loc[hdf["cellid"]==cell_id]["mito_id"].values
print(cell_id)
print(" ".join([str(m) for m in mitoids]))

648518346349539215
3892358 2391819 642563 1235455 3077184 3219479 3073688 3080859 3185207 3072450 3205192 3289098 3217590 2963751 2768078 3083156 2965526 3072709 3273824 3302229 3096090 3065135 3072542 3537118 3184597 3076973 1207782 4069920 3072311 1938692 3305877 3073863 3078411 3188596 1023966 3073112 2556949 3180785 2963562 1553215 3072917 2317286 3102854 2759939 3820803 2618022 3076127 1799732 3073838 3283444 3184925 4182758 2963507 3954099 1708697 3195367 3078605 3536928 3097914 3388173 3325824 3083145 2968405 3083077 855289 2049687 3826907 2829746 1487268 2556937 2963757 1258225 3078668 2968360 3185353 3198087 3305782 3083086 3066639 1569991 3078638 2537657 2542160 2173126 3078420 3078576 3078506 2874829 2192184 3074153 3078570 3530595 3288973 2832312 2980464 4332666 2036952 1570059 3081107 3085117 2858902 2947760 3197945 2766772 3519984 2735935 3078264 3074089 3083144 3448939 3400402 3189391 3185057 2400906 3091545 3082881 2963542 3539593 3095263 3072908 3184511 2146872 3184641

In [20]:
df.keys()

Index(['mito_id', 'cellid', 'compartment', 'x', 'y', 'z',
       'length along skeleton', 'PC1 CA', 'PC1 Circum', 'PC1 Length',
       'PC1 inertia moment', 'PC1 symmetry', 'PC2 CA', 'PC2 Circum',
       'PC2 Length', 'PC2 inertia moment', 'PC2 symmetry', 'PC3 CA',
       'PC3 Circum', 'PC3 Length', 'PC3 inertia moment', 'PC3 symmetry',
       'convex hull SA', 'convex hull compactness', 'convex hull volume',
       'mito CA', 'mito SA', 'mito diam', 'mito size', 'sphericity',
       'cell_type', 'cell_subtype'],
      dtype='object')

In [21]:
df.groupby(by=["cell_type", "compartment"])["length along skeleton"].describe()

count         mean          std         min  \
cell_type  compartment                                                        
glia       Branch              9128.0  1585.819413  2128.532117  121.577776   
           Far                  242.0   808.554763  2808.882763  169.942410   
           Soma                1207.0  1127.005688  1460.670079  135.191760   
inhibitory Axonal              7212.0  1130.875501  1091.611841    0.000000   
           Basal               8663.0  2199.828546  2096.032276    0.000000   
           Somatic            11437.0  1111.056257  2509.114684    0.000000   
pyramidal  Apical             18608.0  5549.158319  9594.057021    0.000000   
           Axonal             11484.0   779.920304   725.055393    0.000000   
           Basal              53318.0  5750.559138  7866.320663    0.000000   
           Somatic            90193.0  1222.199333  1970.421912    0.000000   
           Unknown             1811.0  5294.809242  8252.841086    0.000000   
           Unknown dendritic    658.0  3484.197669  3988.851820    0.000000   

                                      25%          50%          75%  \
cell_type  compartment                                                
glia       Branch              501.071586   896.880873  1850.904625   
           Far                 380.591143   557.715075   788.767362   
           Soma                367.584900   626.543956  1326.053329   
inhibitory Axonal              583.589085   964.570294  1383.589312   
           Basal               900.866338  1591.255663  2796.035196   
           Somatic               0.000000     0.000000     0.000000   
pyramidal  Apical              848.212888  2170.607732  6000.191275   
           Axonal              416.415640   655.048927   972.971025   
           Basal              1127.283828  3106.079515  7257.407015   
           Somatic               0.000000     0.000000  1862.420259   
           Unknown             985.641039  2620.380485  6361.873463   
           Unknown dendritic   845.046453  2159.520353  4494.715195   

                                        max  
cell_type  compartment                       
glia       Branch              45244.842066  
           Far                 43846.661482  
           Soma                17223.139161  
inhibitory Axonal              27961.982908  
           Basal               34469.268709  
           Somatic             25678.579957  
pyramidal  Apical             185523.973188  
           Axonal              35647.144660  
           Basal              176547.088617  
           Somatic             41696.453620  
           Unknown            126157.871588  
           Unknown dendritic   34697.324500